In [4]:
import sys
sys.path.append('../src')
import ast
import pandas as pd
from recommender import (
    build_tfidf_matrix, get_similar_titles,
    genre_based_recommendations, recommend_for_segment_by_similarity
)

catalog = pd.read_csv("../data/processed/netflix_titles_clean.csv")
catalog['genre_list'] = catalog['listed_in'].str.split(', ')

interactions = pd.read_csv("../data/processed/viewing_interactions.csv")
interactions['genre_list'] = interactions['genre_list'].apply(ast.literal_eval)  

user_segments = pd.read_csv("../data/processed/user_segments.csv")  

# --- Build TF-IDF matrix once, reuse for all segments ---
tfidf_matrix, vectorizer = build_tfidf_matrix(catalog)
print("TF-IDF matrix shape:", tfidf_matrix.shape)

# --- Run both recommenders for every segment ---
for segment_name in user_segments['segment'].unique():
    print(f"\n{'='*60}\nSEGMENT: {segment_name}\n{'='*60}")

    genre_recs, top_genre = genre_based_recommendations(segment_name, interactions, user_segments, catalog)
    print(f"\n[Genre-based] Top genre: {top_genre}")
    print(genre_recs)

    sim_recs, seed_title = recommend_for_segment_by_similarity_with_fallback(segment_name, interactions, user_segments, catalog, tfidf_matrix)
    print(f"\n[Similarity-based] Seed title: {seed_title}")
    print(sim_recs)

TF-IDF matrix shape: (8807, 5000)

SEGMENT: Steady Regulars

[Genre-based] Top genre: Stand-Up Comedy
    show_id                                title         genre_list  \
281   s5714      Colin Quinn: The New York Story  [Stand-Up Comedy]   
341   s7561         Natalia Valdebenito: Gritona  [Stand-Up Comedy]   
7     s1173              Loyiso Gola: Unlearning  [Stand-Up Comedy]   
198   s5025              Marlon Wayans: Woke-ish  [Stand-Up Comedy]   
307   s5851  Patton Oswalt: Talking for Clapping  [Stand-Up Comedy]   

     segment_avg_completion  segment_watch_count  
281                0.778650                   26  
341                0.769486                   36  
7                  0.768770                   30  
198                0.760118                   36  
307                0.759704                   29  

[Similarity-based] Seed title: Race to Witch Mountain
     show_id                               title  \
6405   s6406   Calico Critters: A Town of Dreams   
664   

In [2]:
for segment_name in user_segments['segment'].unique():
    segment_users = user_segments[user_segments['segment'] == segment_name]['user_id']
    segment_interactions = interactions[interactions['user_id'].isin(segment_users)]
    title_counts = segment_interactions.groupby('show_id').size()
    print(f"{segment_name}: {len(segment_users)} users, max single-title watch count = {title_counts.max() if len(title_counts) else 0}")

Steady Regulars: 2499 users, max single-title watch count = 77
Churning Users: 983 users, max single-title watch count = 35
Power Bingers: 1025 users, max single-title watch count = 112
New / Cold-Start Users: 493 users, max single-title watch count = 5


In [3]:
def recommend_for_segment_by_similarity_with_fallback(segment_name, interactions_df, user_segments_df, catalog_df, tfidf_matrix, top_n=5):
    recs, seed = recommend_for_segment_by_similarity(segment_name, interactions_df, user_segments_df, catalog_df, tfidf_matrix, top_n)
    if recs.empty:
        
        overall_top = (
            interactions_df.groupby('show_id')['completion_rate']
            .agg(['mean', 'count'])
            .query('count >= 100')
            .sort_values('mean', ascending=False)
            .head(top_n)
            .merge(catalog_df[['show_id', 'title', 'listed_in']], on='show_id')
        )
        return overall_top, "No watch history — showing globally popular titles"
    return recs, seed

In [5]:
cold_recs, cold_seed = recommend_for_segment_by_similarity_with_fallback(
    'New / Cold-Start Users', interactions, user_segments, catalog, tfidf_matrix
)
print(cold_seed)
print(cold_recs)

No watch history — showing globally popular titles
  show_id      mean  count                        title  \
0   s4156  0.847838    101          Up Among  The Stars   
1   s8229  0.845473    100           The Brothers Grimm   
2   s6643  0.837070    116  Dragonheart 3: The Sorcerer   
3   s7258  0.836851    100         La Rosa de Guadalupe   
4   s8129  0.835440    100             Superman Returns   

                                           listed_in  
0     Dramas, International Movies, Sci-Fi & Fantasy  
1               Action & Adventure, Sci-Fi & Fantasy  
2               Action & Adventure, Sci-Fi & Fantasy  
3  Classic & Cult TV, Crime TV Shows, Internation...  
4               Action & Adventure, Sci-Fi & Fantasy  


### Note: Cold Start Fallback

The "New / Cold Start Users" segment has insufficient watch history (max 5 
watches per title) to generate personalized similarity based recommendations. 
For this segment, the system falls back to recommending globally popular, 
high completion titles instead a standard approach in real recommender 
systems for users without enough interaction data to personalize against.